In [ ]:
%load_ext autoreload
%autoreload 2

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import config
from instruments.vna import VNA
from instruments.hdawg import HDAWG
from instruments.attenuator import Attenuator
from fitting_algorithms.lorentzian import Lorentzian

In [ ]:
class QubitScan:    
    def __init__(self, readout_p, readout_f, raw_data_dir, plots_dir):    
        self.hdawg = HDAWG(device_serial="DEV8488", host="localhost")
        self.vna = VNA(vna_ip=config.VNA_IP, vna_port=config.VNA_PORT)
        self.atten = Attenuator(config.LABBER_SERVER, config.LABBER_SERVER_TIMEOUT, config.ATTENUATOR_NAME, config.ATTENUATOR_INTERFACE, config.ATTENUATOR_ADDRESS)
        self.atten_wait = config.ATTENUATOR_WAIT_S
        self.atten.set_attenuation(readout_p, wait=self.atten_wait)
        self.plots_dir = plots_dir
        self.readout_f = readout_f
        self.vna.power_dbm = config.VNA_POWER_DBM
        self.vna.ifbw = config.VNA_IFBW
        self.vna.averages = config.VNA_AVERAGES
        self.scan_counter = 0
        self.naive = True
        self.raw_data_dir = raw_data_dir

    def measure_s21(self, freq, power):
        self.hdawg.power_dbm = power; assert power <= 0
        self.hdawg.on()
        self.hdawg.freq = freq
        self.hdawg._prepare_cw_waveform()
        s21 = self.vna.measure_cw_point()
        return s21
    
    def scan_linspace(self, new_freqs=None):
        self.hdawg.off()
        time_start = time.time()
        freqs, s21, s21_background = np.array([]), np.array([]), np.array([])
        self.vna.prepare_for_cw_sweep(self.readout_f)
        
        sweep_freqs = new_freqs if new_freqs is not None else np.linspace(config.SG_START_FREQ, config.SG_STOP_FREQ, config.SG_DENSE_SWEEP_POINTS if self.naive else config.SG_COARSE_SWEEP_POINTS)
        
        # Background S21 measurement
        self.vna.prepare_for_cw_sweep(self.readout_f)
        for f1 in sweep_freqs:
            self.hdawg.set_cw_tone(f1)
            time.sleep(config.SG_VNA_WAIT_TIME)
            s21__background_point = self.vna.measure_cw_point()
            s21_background = np.append(s21_background, s21__background_point)
        raw_data_filename_bg = f"raw_{self.hdawg.power_dbm}dBm_dense_background.npz" if self.naive else f"raw_{self.hdawg.power_dbm}dBm_coarse_background.npz"
        
        self.vna.finalize_sweep()
        # S21 measurement
        self.hdawg.on()
        for f in sweep_freqs:
            self.hdawg.set_cw_tone(f)
            time.sleep(config.SG_VNA_WAIT_TIME)
            s21_point = self.vna.measure_cw_point()
            s21 = np.append(s21, s21_point)
            freqs = np.append(freqs, f)
        raw_data_filename = f"raw_{self.hdawg.power_dbm}dBm_dense.npz" if self.naive else f"raw_{self.hdawg.power_dbm}dBm_coarse.npz"
        self.hdawg.off()
        
        self.vna.finalize_sweep()
    

        print(f"TIME TAKEN: {time.time() - time_start:.2f}s")
        
        raw_data_path = os.path.join(self.raw_data_dir, raw_data_filename)
        np.savez(raw_data_path, freqs=np.array(freqs), s21=np.array(s21))

        raw_data_path_bg = os.path.join(self.raw_data_dir, raw_data_filename_bg)
        np.savez(raw_data_path_bg, freqs=np.array(freqs), s21=np.array(s21_background))

        self.scan_counter += 1
        return np.array(freqs), np.array(s21), np.array(s21_background)
    
    def scan_adaptive(self):
        f, s_21, s_21_bg = self.scan_linspace()
        s_21_bg += np.zeros_like(s_21) # adaptive lorentzian fitting does not require background subtraction
        return f, s_21


In [ ]:
def run(readout_p, readout_f, raw_data_dir, plots_dir):
    qubit_scan = QubitScan(readout_p, readout_f, raw_data_dir, plots_dir)
    start = time.time()
    lorentzian = Lorentzian(
            naive=qubit_scan.naive,
            adaptive_sweep_points=config.SG_ADAPTIVE_SWEEP_POINTS,
            adaptive_iterations=config.SG_ADAPTIVE_ITERATIONS,
            fetch_data=qubit_scan.measure_s21
        )
        
    power_sweep_results = []
    for power in config.SG_POWERS:
        print(f"--- SG Power {power} dBm ---\n")
        qubit_scan.hdawg.set_power(power, config.SG_MAX_ALLOWED_POWER)
        qubit_scan.scan_counter = 0
            
        f0, kappa, plots_dict, cleaned_data_dict = lorentzian.fitting_routine()
        power_sweep_results.append((power, f0, kappa))
            
        for iteration_name, fig in plots_dict.items():
            plot_filename = f"fitted_{power}dBm_{iteration_name}.png"
            plots_path = os.path.join(qubit_scan.plots_dir, plot_filename)
            plt.savefig(plots_path, dpi=150)
            plt.close(fig)

    results_array = np.array(power_sweep_results)
    fitted_powers = results_array[:, 0]
    fitted_f0s = results_array[:, 1]
    fitted_kappas = results_array[:, 2]

    baseline_kappa = np.mean(fitted_kappas[:3])
    threshold_kappa = baseline_kappa * config.THRESHOLD_KAPPA_MULTIPLIER
      
    broadened_indices = np.where(fitted_kappas > threshold_kappa)[0]
        
    if len(broadened_indices) > 0:
        optimal_idx = max(0, broadened_indices[0] - 1)
    else:
        optimal_idx = -1 
            
    drive_power = fitted_powers[optimal_idx]
    drive_frequency = fitted_f0s[optimal_idx]
        
    print(f"\n--- OPTIMAL DRIVE FOUND ---")
    print(f"Freq: {drive_frequency/1e9:.5f} GHz | Power: {drive_power} dBm")

    stop = time.time()
    print(f"Time taken: {(stop-start):.2f}s\n")        
    return drive_frequency, drive_power